# Fantrax API Wrapper: Comprehensive Feature Guide & Analysis Notebook

This notebook demonstrates how to access and analyze Fantrax Fantasy Premier League data using the clean, modular `fantrax_wrapper` package.

### Capabilities Covered:
1. **Environment Configuration**: Loading credentials and league settings from `.env`.
2. **League & Schedule Queries**: Fetching head-to-head weekly matchups and cumulative standings.
3. **Roster Management**: Extracting starting XIs, bench reserves, and injured reserve for any team.
4. **Player Database & Box Scores**: Querying player stats, historical game logs, and latest news.
5. **Live Scoring & Projections**: Fetching live scoring and projected points per week.
6. **Position-Specific Scoring Engine**: Calculating FPts and Adjusted FPts (aFPts) across positions (G, D, M, F).
7. **Manager Efficiency & Optimal Standings**: Optimal Starting XI solver, lineup inefficiency, counterfactual optimal standings, and standings against weekly median.
8. **Waiver Wire Evaluation**: Filtering and ranking available free agents across real league rosters.
9. **Stat Normalization**: Per-90 stat metrics and trimmed mean opponent baselines.
10. **Raw RPC Queries**: Custom Fantrax RPC methods.
11. **Custom Scoring Rules**: Positional scoring weight engines.
12. **Data Export**: CSV and JSON reporting.


## 1. Setup & Environment Configuration
Sourcing environment variables from  and initializing .

In [36]:
import os
import sys
import importlib
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Ensure parent directory is in Python path
sys.path.insert(0, os.path.abspath(".."))

import fantrax_wrapper
import fantrax_wrapper.models
import fantrax_wrapper.models.team
import fantrax_wrapper.client
import fantrax_wrapper.rpc
import fantrax_wrapper.analytics
import fantrax_wrapper.scoring

importlib.reload(fantrax_wrapper.models.team)
importlib.reload(fantrax_wrapper.models)
importlib.reload(fantrax_wrapper.rpc)
importlib.reload(fantrax_wrapper.scoring)
importlib.reload(fantrax_wrapper.analytics)
importlib.reload(fantrax_wrapper.client)
importlib.reload(fantrax_wrapper)

from fantrax_wrapper import (
    FantraxClient,
    FantasyScoringEngine,
    calculate_manager_efficiency,
    calculate_median_standings,
    evaluate_waiver_candidates,
    normalize_per_90_stats,
    compute_positional_opponent_averages,
    load_env_file,
)

# Load environment variables from soccer/config/.env
env_loaded = load_env_file("../config/.env")
print(f"Loaded environment file: {env_loaded}")

cookie_path = "fantraxloggedin.cookie"
client = FantraxClient(cookie_path=cookie_path, auto_login=True)
print(f"FantraxClient initialized for League ID: {client.league_id}")
print(f"Client has get_draft_rankings: {hasattr(client, 'get_draft_rankings')}")


Loaded environment file: True
FantraxClient initialized for League ID: iku2an1nmryzcfa2
Client has get_draft_rankings: True


In [37]:
CURRENT_WEEK = 1

## 2. Matchup Schedule & League Table
Fetching head-to-head match schedules, weekly scores, and calculating official league table standings.

In [38]:
# Fetch weekly head-to-head matchups
matchups_df = client.get_matchups(through_week=CURRENT_WEEK, as_dataframe=True)
print(f"Fetched {len(matchups_df)} matchup rows.")
matchups_df.head(14)


Fetched 14 matchup rows.


,week,team,team_id,score,opponent,opponent_score,matchupId,result
0,1,FC Purulona,4nxisu5rmskmmet5,65.25,Estimated Profit,45.25,0,W
1,1,Estimated Profit,gxu0fnbomsgg77tu,45.25,FC Purulona,65.25,0,L
2,1,youngmoon,lgwrbwiomsnz7zra,115.25,sduvuuru,76.50,1,W
3,1,sduvuuru,0ogxe7tgmskrzfjc,76.50,youngmoon,115.25,1,L
4,1,Wallalujah,xhzzcufsmsmqo7v0,116.75,Arnie-senal,97.00,2,W
5,1,Arnie-senal,089oci7tmsguca8i,97.00,Wallalujah,116.75,2,L
6,1,seanbridges7,vyrycfi0mssfpf53,107.50,MOO FC,97.00,3,W
7,1,MOO FC,584rnmlnmsgnf79q,97.00,seanbridges7,107.50,3,L
8,1,Smip Estonian,kvt35nm5msggps39,74.00,WayneRooney10,65.75,4,W
9,1,WayneRooney10,ouf8h7w1mskuaeh3,65.75,Smip Estonian,74.00,4,L


In [39]:
# Fetch overall league table standings
standings_df = client.get_league_table(through_week=CURRENT_WEEK)
print("Official League Table:")
standings_df


Official League Table:


,wins,draws,losses,record,points,fpts_for,fpts_against,avg_fpts_for,avg_fpts_against
team,,,,,,,,,
Wallalujah,1,0,0,1-0-0,3,116.75,97.00,116.75,97.00
youngmoon,1,0,0,1-0-0,3,115.25,76.50,115.25,76.50
FC VAR,1,0,0,1-0-0,3,113.50,46.75,113.50,46.75
seanbridges7,1,0,0,1-0-0,3,107.50,97.00,107.50,97.00
Birds & Bees FC,1,0,0,1-0-0,3,88.75,73.75,88.75,73.75
Smip Estonian,1,0,0,1-0-0,3,74.00,65.75,74.00,65.75
FC Purulona,1,0,0,1-0-0,3,65.25,45.25,65.25,45.25
Arnie-senal,0,0,1,0-0-1,0,97.00,116.75,97.00,116.75
MOO FC,0,0,1,0-0-1,0,97.00,107.50,97.00,107.50


## 3. Team Roster Breakdown
Extracting team roster breakdown (Starting XI, Reserves, Injured Reserve) for a given week.

In [40]:
# Fetch roster for an active team
if not matchups_df.empty:
    sample_team_id = matchups_df["team_id"].iloc[0]
    sample_team_name = matchups_df["team"].iloc[0]
    roster = client.get_team_roster(team_id=sample_team_id, period=CURRENT_WEEK)
    
    print(f"Roster Breakdown for {sample_team_name} (Team ID: {sample_team_id}) - Week {CURRENT_WEEK}:")
    print(f"-> Starters: {len(roster.starters)} players")
    print(f"-> Reserves: {len(roster.reserves)} players")
    print(f"-> Injured Reserve: {len(roster.injured_reserve)} players\n")
    
    roster_df = pd.DataFrame([vars(r) for r in (roster.starters + roster.reserves)])
    display(roster_df[["player_name", "position_short", "is_starter", "is_reserve", "points"]].head(16))


Roster for Team ID s3sntuejmryzsnsg (Week 1):
Goalkeeper: Bart Verbruggen
Defenders: ['Danny Ballard', 'Reece James', 'Harry Maguire', 'Ben White']
Midfielders: ['Pascal Gross', 'Dwight McNeil', 'Anton Stach']
Forwards: ['Joao Pedro', 'Rio Ngumoha', 'Christos Tzolis']
Reserves (Bench): ['Jeremie Frimpong', 'Moises Caicedo', 'Tammy Abraham', 'Promise David', 'Nicolas Jackson']
Injured Reserve (IR): ['Jurrien Timber']


## 4. Player Database, Game Logs & News
Querying the player database, inspecting detailed game logs for individual players, and fetching news updates.

In [41]:
# Fetch top players database
players_df = client.get_all_players(as_dataframe=True)
print(f"Total players fetched: {len(players_df)}")
players_df[["name", "teamShortName", "posShortNames", "fpts", "fpts_per_game", "next_match"]].head(10)


Total players fetched: 620


,name,teamShortName,posShortNames,fpts,fpts_per_game,next_match
0,Maxim De Cuyper,BHA,D,35.00,35.00,@CHE<br/>Sun 9:00AM
1,Cole Palmer,CHE,M,31.00,31.00,BHA<br/>Sun 9:00AM
2,Jack Hinshelwood,BHA,M,28.50,28.50,@CHE<br/>Sun 9:00AM
3,Michael Kayode,BRF,D,26.25,26.25,@LEE<br/>Sun 9:00AM
4,Morgan Rogers,CHE,"M,F",25.50,25.50,BHA<br/>Sun 9:00AM
5,Konstantinos Tzolakis,HUL,G,25.00,25.00,@COV<br/>Sat 10:00AM
6,Semi Ajayi,HUL,D,24.25,24.25,@COV<br/>Sat 10:00AM
7,Nobel Mendy,HUL,D,24.00,24.00,@COV<br/>Sat 10:00AM
8,Cody Gakpo,LIV,F,23.50,23.50,NOT<br/>Sat 7:30AM
9,Ben White,ARS,D,22.75,22.75,@AVL<br/>Mon 3:00PM


In [42]:
# Query game log for a top player
if not players_df.empty:
    sample_player_id = players_df.iloc[0]["scorerId"]
    sample_player_name = players_df.iloc[0]["name"]
    print(f"Fetching game log for: {sample_player_name} (ID: {sample_player_id})")
    game_log_df = client.get_player_game_log(player_id=sample_player_id, year=2026)
    display(game_log_df.head(10))

Fetching game log for: Maxim De Cuyper (ID: 06y9m)


,Date,Team,Opp,Score,FPts,Min,G,KP,AT,SOT,...,AER,PKM,PKD,OG,GAO,CS,player_id,Date_dt,opp_clean,is_home
0,Aug 23,BHA,AVL,W 4-0,35,77,1,3,1,1,...,0,0,0,0,0,1,06y9m,2026-08-23,AVL,True


In [43]:
# Fetch latest news for a top player
if not players_df.empty:
    sample_player_id = players_df.iloc[0]["scorerId"]
    sample_player_name = players_df.iloc[0]["name"]
    news = client.get_player_news(player_id=sample_player_id)
    print(f"Latest News for {sample_player_name}:")
    print(news if news else "No recent news analysis text.")

Latest News for Maxim De Cuyper:
De Cuyper scored once in Sunday's 4-0 win over Aston Villa, while his early effort resulted in Victor Lindelof's own goal before he found the net himself from close range later in the first half. He finished with four shots (one on target), five crosses and two corners, highlighting his consistent involvement in the final third. De Cuyper's forward runs and ability to get into dangerous areas make him a significant attacking threat from the left flank and should provide opportunities for further goal contributions.


## 5. Live Scoring & Projections
Fetching live scoring stats and projected fantasy points map for a scoring period.

In [44]:
# Fetch live scoring & projections for Week 1
live_scoring_df = client.get_live_scoring(period=1)
print(f"Fetched {len(live_scoring_df)} projection entries for Week 1:")
live_scoring_df.head(10)


Fetched 150 projection entries for Week 1:


,teamId,status,scorerId,fantrax_projection,week
0,3gw7nm5tmsghshpe,ACTIVE,05u1m,3.69,1
1,3gw7nm5tmsghshpe,ACTIVE,06fxe,5.89,1
2,3gw7nm5tmsghshpe,ACTIVE,04sjt,4.08,1
3,3gw7nm5tmsghshpe,ACTIVE,06uyc,0.29,1
4,3gw7nm5tmsghshpe,ACTIVE,0427d,6.32,1
5,3gw7nm5tmsghshpe,ACTIVE,068a0,6.90,1
6,3gw7nm5tmsghshpe,ACTIVE,067yz,10.40,1
7,xhzzcufsmsmqo7v0,ACTIVE,0639x,6.99,1
8,xhzzcufsmsmqo7v0,ACTIVE,06cmm,7.86,1
9,xhzzcufsmsmqo7v0,ACTIVE,067ys,5.75,1


## 6. Position-Specific Fantasy Scoring Engine
Calculating fantasy points (FPts) and adjusted fantasy points (aFPts) using position-specific scoring rules (GK vs D, M, F).

In [45]:
scoring_engine = FantasyScoringEngine()

# Calculate FPts and aFPts using real match box score lines from the player database
if not players_df.empty:
    mid_player = players_df[players_df["posShortNames"].str.contains("M", na=False)].iloc[0]
    gk_player = players_df[players_df["posShortNames"].str.contains("G", na=False)].iloc[0]
    
    mid_log = client.get_player_game_log(player_id=mid_player["scorerId"])
    gk_log = client.get_player_game_log(player_id=gk_player["scorerId"])
    
    if not mid_log.empty:
        mid_stats = mid_log.iloc[0].to_dict()
        mid_fpts = scoring_engine.calculate_fantasy_score(mid_stats, position="M")
        mid_afpts = scoring_engine.calculate_adjusted_fantasy_score(mid_stats, position="M")
        print(f"Real Match Box Score for Midfielder: {mid_player['name']} vs {mid_stats.get('Opp', '')}")
        print(f"-> Calculated FPts: {mid_fpts} | aFPts (non-luck baseline): {mid_afpts}\n")
        
    if not gk_log.empty:
        gk_stats = gk_log.iloc[0].to_dict()
        gk_fpts = scoring_engine.calculate_fantasy_score(gk_stats, position="G")
        gk_afpts = scoring_engine.calculate_adjusted_fantasy_score(gk_stats, position="G")
        print(f"Real Match Box Score for Goalkeeper: {gk_player['name']} vs {gk_stats.get('Opp', '')}")
        print(f"-> Calculated FPts: {gk_fpts} | aFPts (non-luck baseline): {gk_afpts}")


Midfielder Line: {'G': 1, 'AT': 1, 'KP': 3, 'SOT': 2, 'TKW': 2, 'DIS': -1, 'YC': -2}
-> FPts: 31.5 | aFPts (non-luck): 16.5
Goalkeeper Line: {'CS': 1, 'SV': 4, 'GAO': 0, 'PKS': 0, 'DIS': 0}
-> FPts: 16.0 | aFPts (non-luck): 16.0


## 7. Manager Efficiency & Optimal Lineup Analytics
Solving for optimal starting XIs (), measuring lineup inefficiency, and computing standings against weekly median score.

In [46]:
# Calculate real manager efficiency across completed matchweeks
efficiency_df = client.get_league_manager_efficiency(through_week=CURRENT_WEEK)

print("Manager Lineup Efficiency Metrics (Ranked by Optimal Potential Score):")
display(efficiency_df.sort_values("avg_max_score", ascending=False))


Manager Lineup Efficiency Metrics:


,avg_inefficiency,total_missed_points,avg_actual_score,avg_max_score
team,,,,
Traderjoe18,3.00,3.00,46.75,49.75
Estimated Profit,7.50,7.50,45.25,52.75
Smip Estonian,3.00,3.00,74.00,77.00
FC Purulona,14.00,14.00,65.25,79.25
sduvuuru,8.75,8.75,76.50,85.25
WayneRooney10,22.75,22.75,65.75,88.50
Birds & Bees FC,3.00,3.00,88.75,91.75
Arnie-senal,5.25,5.25,97.00,102.25
seanbridges7,2.75,2.75,107.50,110.25


In [47]:
if not matchups_df.empty:
    median_standings_df = calculate_median_standings(matchups_df, only_completed=True)
    print("Standings Evaluated Against Weekly League Median Score:")
    display(median_standings_df)


Standings Evaluated Against Weekly League Median:


### Counterfactual Optimal Standings
Simulates what the league table and match outcomes would look like if every manager had set their optimal Starting XI each week, highlighting bench optimization and lineup luck deltas.

In [ ]:
# 1. Fetch Simulated Counterfactual Optimal League Standings
optimal_standings_df = client.get_optimal_standings(through_week=CURRENT_WEEK)
print("Simulated Optimal League Table (If all managers set optimal Starting XIs):")
display(optimal_standings_df)

# 2. Compare Actual Standings vs Optimal Standings (Rank & Points Delta)
if not standings_df.empty and not optimal_standings_df.empty:
    from fantrax_wrapper import compare_standings_vs_optimal
    comparison_df = compare_standings_vs_optimal(standings_df, optimal_standings_df)
    print("\nActual Standings vs Optimal Potential Comparison (Rank & Points Delta):")
    display(comparison_df[["actual_rank", "optimal_rank", "rank_delta", "record_actual", "record_optimal", "pts_delta", "missed_fpts"]])


## 8. Waiver Wire Candidate Evaluation
Filtering available free agents by minutes played, average FPts per game, and position.

In [48]:
# Collect all rostered player names across all teams in the league
all_rostered_names = []
if not matchups_df.empty:
    unique_team_ids = matchups_df["team_id"].unique()
    for t_id in unique_team_ids:
        team_roster = client.get_team_roster(team_id=str(t_id), period=CURRENT_WEEK)
        all_players_on_team = (
            ([team_roster.goalkeeper] if team_roster.goalkeeper else [])
            + team_roster.defenders
            + team_roster.midfielders
            + team_roster.forwards
            + team_roster.reserves
            + team_roster.injured_reserve
        )
        for p in all_players_on_team:
            if p and p.player_name:
                all_rostered_names.append(p.player_name)

all_rostered_names = list(set(all_rostered_names))
print(f"Rostered players identified across league: {len(all_rostered_names)}")

# Evaluate free agent waiver candidates using real league rosters
if not players_df.empty:
    waivers_df = evaluate_waiver_candidates(
        df_player_stats=players_df,
        rostered_player_names=all_rostered_names,
        current_week=CURRENT_WEEK,
        min_last_week_minutes=0,
        exclude_goalkeepers=True
    )
    print("Top 10 Available Waiver Wire Candidates:")
    display(waivers_df[["name", "teamShortName", "posShortNames", "fpts", "fpts_per_game"]].head(10))


Evaluated 610 free agent waiver candidates:


,name,teamShortName,posShortNames,fpts,fpts_per_game
0,Joao Pedro,CHE,F,22.00,22.00
1,Emersonn,IPS,F,22.00,22.00
2,Mamadou Sangaré,BRF,M,22.00,22.00
3,Vitaly Janelt,BRF,M,21.50,21.50
4,Timothy Castagne,FUL,D,20.75,20.75
5,Dominik Szoboszlai,LIV,M,20.50,20.50
6,Martin Odegaard,ARS,M,20.00,20.00
7,Rayan Cherki,MCI,M,20.00,20.00
8,Josko Gvardiol,MCI,D,19.25,19.25
9,Anton Stach,LEE,M,18.50,18.50


## 9. Stat Normalization & Positional Opponent Averages
Normalizing counting statistics to per-90 metrics and computing trimmed mean positional baselines against opponents.

In [49]:
# 1. Fetch and aggregate real match logs for top active league players
logs = []
if not players_df.empty:
    for _, p in players_df.head(15).iterrows():
        df_log = client.get_player_game_log(player_id=p["scorerId"])
        if not df_log.empty:
            df_log["player_name"] = p["name"]
            df_log["position"] = p.get("posShortNames", "M")
            logs.append(df_log)

if logs:
    real_match_logs = pd.concat(logs, ignore_index=True)
    
    # 2. Normalize counting stats to Per-90 rates
    norm_df = normalize_per_90_stats(
        df=real_match_logs,
        stat_columns=["G", "KP", "SOT", "TkW", "Int", "CLR", "FPts"],
        minutes_col="Min"
    )
    print(f"Per-90 Normalized Performance Stats ({len(norm_df)} match logs):")
    display(norm_df[["player_name", "position", "opp_clean", "Min", "FPts", "FPts_per_90", "KP_per_90", "TkW_per_90"]].head(10))


Per-90 Normalized Performance Stats for Maxim De Cuyper:


,Date,Opp,Min,FPts,FPts_per_90,KP_per_90,TkW_per_90,SOT_per_90
0,May 24,MUN,45,5,9.99,6.03,0.00,0.00
1,May 17,@LEE,90,8.25,8.28,1.98,0.00,0.00
2,May 9,WOL,87,36.25,37.53,6.21,0.00,0.00
3,May 2,@NEW,9,1.25,12.51,9.99,9.99,0.00
4,Apr 21,CHE,13,12,83.07,6.93,6.93,0.00
5,Apr 18,@TOT,15,-1.75,-10.53,0.00,0.00,0.00
6,Mar 14,@SUN,13,1,6.93,0.00,0.00,0.00
7,Mar 1,NOT,1,0.25,22.50,0.00,0.00,0.00
8,Feb 21,@BRF,13,1,6.93,0.00,0.00,0.00
9,Feb 8,CRY,70,0.25,0.36,0.00,0.00,1.26


In [50]:
if 'norm_df' in locals() and not norm_df.empty:
    # 3. Calculate trimmed-mean defensive allowances per opponent team and position
    opp_averages_df = compute_positional_opponent_averages(
        df=norm_df,
        per90_columns=["KP_per_90", "SOT_per_90", "TkW_per_90", "FPts_per_90"],
        opp_col="opp_clean",
        pos_col="position"
    )
    print("Positional Opponent Trimmed Mean Averages Conceded:")
    display(opp_averages_df.head(15))


KeyError: 'opp'

## 10. Raw League Metadata & Custom RPC Queries
`FantraxClient.get_raw_league_info()` provides low-level access to any Fantrax RPC method (`getLeagueInfo`, `getLeagueRules`, `getDraftResults`, `getTransactionHistory`).

In [ ]:
# Query raw league metadata and rules
raw_info = client.get_raw_league_info(method="getLeagueInfo")
league_name = raw_info.get("name", "Fantrax Premier League")
print(f"Raw League Name: {league_name}")

# Query raw draft results / transactions
draft_info = client.get_raw_league_info(method="getDraftResults")
print(f"Draft Results payload keys: {list(draft_info.keys())[:5] if isinstance(draft_info, dict) else type(draft_info)}")


## 11. Custom Fantasy Scoring Engine Configuration
The `FantasyScoringEngine` supports custom positional point allocations per stat event (Goal, Assist, Clean Sheet, Tackle, Interception, Save, Key Pass).

In [ ]:
from fantrax_wrapper.scoring import FantasyScoringEngine

# Initialize FantasyScoringEngine with custom positional scoring weights
custom_scoring_weights = {
    "D": {"G": 10.0, "AT": 7.0, "CS": 6.0, "TKW": 1.0, "KP": 2.0, "INT": 1.0},
    "M": {"G": 9.0, "AT": 7.0, "TKW": 1.0, "KP": 2.0, "INT": 1.0},
    "F": {"G": 8.0, "AT": 7.0, "KP": 2.0, "SOT": 2.0},
}

custom_engine = FantasyScoringEngine(custom_rules=custom_scoring_weights)

# Compare custom scoring rules against standard scoring using a real match box score
if not players_df.empty:
    top_mid = players_df[players_df["posShortNames"].str.contains("M", na=False)].iloc[0]
    mid_log = client.get_player_game_log(player_id=top_mid["scorerId"])
    if not mid_log.empty:
        real_stats = mid_log.iloc[0].to_dict()
        custom_fpts = custom_engine.calculate_fantasy_score(real_stats, position="M")
        standard_fpts = scoring_engine.calculate_fantasy_score(real_stats, position="M")
        print("Custom Fantasy Scoring Engine Initialized.")
        print(f"Real Match Line: {top_mid['name']} vs {real_stats.get('Opp', '')}")
        print(f"-> Standard Scoring FPts: {standard_fpts} pts")
        print(f"-> Custom Rule Set FPts:  {custom_fpts} pts")


## 12. Data Export & CSV / JSON Integration
All tabular data returned by `FantraxClient` can be exported directly to CSV or JSON for downstream modeling.

In [ ]:
# Export league standings to CSV
standings_df = client.get_league_table()
if not standings_df.empty:
    csv_path = "league_standings.csv"
    standings_df.to_csv(csv_path, index=True)
    print(f"Exported league standings to {csv_path}")

# Export top players to JSON
players_df = client.get_all_players(as_dataframe=True)
if not players_df.empty:
    json_path = "top_players.json"
    players_df.head(20).to_json(json_path, orient="records", indent=2)
    print(f"Exported top 20 players to {json_path}")


## 13. Optimal Lineup Optimizer (Given Team & Matchweek)
`FantraxClient.get_optimal_lineup(team_id, period)` calculates the maximum possible Starting XI fantasy score and individual player lineup selection for any team in a given week.

In [ ]:
# Calculate optimal lineup for an active team in CURRENT_WEEK
matchups_df = client.get_matchups(through_week=CURRENT_WEEK, only_completed=True, as_dataframe=True)
if not matchups_df.empty:
    sample_team = matchups_df.iloc[0]
    sample_team_id = sample_team["team_id"]
    sample_team_name = sample_team["team"]
    optimal_info = client.get_optimal_lineup(team_id=sample_team_id, period=CURRENT_WEEK)
    
    print(f"Optimal Lineup Analysis for {sample_team_name} (Week {CURRENT_WEEK}):")
    print(f"  Actual Score Achieved: {optimal_info['actual_score']} pts")
    print(f"  Max Optimal Score Possible: {optimal_info['optimal_score']} pts")
    print(f"  Manager Lineup Efficiency: {optimal_info['efficiency_pct']}%\n")
    
    starters_df = pd.DataFrame(optimal_info["starters"])
    print(f"Optimal Starting XI ({len(starters_df)} players):")
    display(starters_df[["player_name", "position", "score"]])


## 14. Draft Rankings & Player ADP Projections
`FantraxClient.get_draft_rankings(season_or_projection, league_id)` pulls the complete draft rankings list, projected fantasy points, points per game, and percentage drafted.

In [ ]:
# Fetch draft rankings list
projections_2026_df = client.get_draft_rankings(season_or_projection="PROJECTION_0_926_SEASON")

print(f"Fetched {len(projections_2026_df)} draft ranked players:")
display(projections_2026_df.head(15))


In [ ]:
# Fetch draft rankings list
performance_2026_df = client.get_draft_rankings(season_or_projection="SEASON_925_YEAR_TO_DATE")
performance_2026_df = performance_2026_df.rename(columns={'total_pts': '2026_total_pts'})
print(f"Fetched {len(performance_2026_df)} draft ranked players:")
display(performance_2026_df.head(15))


In [ ]:
# Fetch draft rankings list
performance_2024_df = client.get_draft_rankings(season_or_projection="SEASON_924_YEAR_TO_DATE")
performance_2024_df = performance_2024_df.rename(columns={'total_pts': '2024_total_pts'})
print(f"Fetched {len(performance_2024_df)} draft ranked players:")
display(performance_2024_df.head(15))


In [ ]:
if not projections_2026_df.empty and not performance_2026_df.empty:
    draft_df = pd.merge(left=projections_2026_df, right=performance_2026_df, on=['player_id', 'name'], how='left', suffixes=['', '_2026'])
else:
    draft_df = pd.DataFrame()


In [ ]:
if not draft_df.empty and not performance_2024_df.empty:
    draft_df = pd.merge(left=draft_df, right=performance_2024_df, on=['player_id', 'name'], how='left', suffixes=['', '_2024'])


In [ ]:
if not draft_df.empty:
    selected_cols = [c for c in ['name', 'adp', 'team_name', 'position', 'projected_pts', '2026_total_pts', '2024_total_pts'] if c in draft_df.columns]
    draft_df = draft_df[selected_cols]
    display(draft_df.head(10))
else:
    print("Draft data is consolidated when draft rankings are active.")


In [ ]:
if not draft_df.empty:
    draft_df.to_csv('draft_data.csv', index=False)
    print("Exported draft_data.csv successfully.")
